## Note from Teacher

We will need to setup searching for Linear, Binary, and Hash.   We should talk about all of these and the best data structure choice for searching.  

The data below sorts Objects, you should focus on Arrays, ArrayLists and Hashmaps.  Then you could conclude with the object examples provided.

## Foundation for Dynamic Key-Based Sorting
> Implementing an abstract class using `Comparable` to enable dynamic key selection and sorting, demonstrated with a sample data set.

In [128]:
/**
 * [TEACHING GUIDE] Foundation for Sortable Collections
 * 
 * ARCHITECTURE:
 * This class acts as a blueprint for sortable objects. By implementing 
 * {@link Comparable}, we establish a "sorting contract" that allows 
 * objects to be organized automatically.
 *
 * CORE COMPONENTS TO IMPLEMENT:
 * 
 * • DATA CATEGORY: Define a {@code type} to identify the subclass.
 * • KEY DEFINITION: Use the {@link KeyTypes} interface to declare sortable attributes.
 * • KEY SELECTION: Implement {@link #getKey()} to provide the active sorting criteria.
 * • DYNAMIC OUTPUT: Implement {@link #toString()} to reflect the current key in the output.
 *
 * GOAL: 
 * Understand how a single class can change its identity and sort order 
 * dynamically by simply changing its internal state.
 */
public abstract class Collectable implements Comparable <Collectable> {
	public final String masterType = "Collectable";
	private String type;	// extender should define their data type

	// enumerated interface
	public interface KeyTypes {
		String name();
	}
	protected abstract KeyTypes getKey();  	// this method helps force usage of KeyTypes

	// getter
	public String getMasterType() {
		return masterType;
	}

	// getter
	public String getType() {
		return type;
	}

	// setter
	public void setType(String type) {
		this.type = type;
	}
	
	// this method is used to establish key order
	public abstract String toString();

	// this method is used to compare toString of objects
	public int compareTo(Collectable obj) {
		return this.toString().compareTo(obj.toString());
	}

	// static print method used by extended classes
	public static void print(Collectable[] objs) {
		// print 'Object' properties
		System.out.println(objs.getClass() + " " + objs.length);

		// print 'Collectable' properties
		if (objs.length > 0) {
			Collectable obj = objs[0];	// Look at properties of 1st element
			System.out.println(
					obj.getMasterType() + ": " + 
					obj.getType() +
					" listed by " +
					obj.getKey());
		}

		// print "Collectable: Objects'
		Collections.sort(Arrays.asList(objs));
		for(Object o : objs)	// observe that type is Opaque
			System.out.println(o);

		System.out.println();
	}
}

In [ ]:
import java.util.Collections;
import java.util.HashMap;
import java.util.Map;
import java.util.Set;
import java.util.StringJoiner;

/* Catalog keeps Set properties
 * each key is stored once; a second add does not replace the stored value
 */
public class Catalog {
    /* catalog entry
     * keeps the key with its properties when passed to an Animal
     */
    public record Entry(String key, Map<String, Object> value) {
        /* string representation
         * returns this entry as {key, value}, without printing
         */
        @Override
        public String toString() {
            return "{" + key + ", " + value + "}";
        }
    }

    // unique keys; order is not guaranteed, the same rule as HashSet
    private final Map<String, Map<String, Object>> catalog = new HashMap<>();

    /* add refuses a duplicate key
     * returns true when the key was added, false when it was already present
     */
    public boolean add(String key, Map<String, Object> value) {
        // putIfAbsent leaves an existing key unchanged
        return catalog.putIfAbsent(key, value) == null;
    }

    /* getter
     * returns the key and properties, or null when the key is missing
     */
    public Entry get(String key) {
        Map<String, Object> value = catalog.get(key);
        return value == null ? null : new Entry(key, value);
    }

    /* Set contains
     * true when the key is already present
     */
    public boolean contains(String key) {
        return catalog.containsKey(key);
    }

    /* key set
     * Map keys are a Set: no duplicates, and order is not guaranteed
     */
    public Set<String> keys() {
        return Collections.unmodifiableSet(catalog.keySet());
    }

    /* string representation
     * returns each entry's string representation on its own line
     */
    @Override
    public String toString() {
        StringJoiner entries = new StringJoiner(System.lineSeparator());
        for (Map.Entry<String, Map<String, Object>> entry : catalog.entrySet()) {
            entries.add(new Entry(entry.getKey(), entry.getValue()).toString());
        }
        return entries.toString();
    }

    // Create and populate catalog for animal names and properties
    public static Catalog animalSet = new Catalog();
    static {
        animalSet.add("lion", Map.of("baby", "cub", "adult", 8));
        animalSet.add("pig", Map.of("baby", "piglet", "adult", 3));
        animalSet.add("robin", Map.of("baby", "chick", "adult", 1));
        animalSet.add("cat", Map.of("baby", "kitten", "adult", 2));
        animalSet.add("dog", Map.of("baby", "puppy", "adult", 2));
    }

    /* main to test Catalog Set properties
     * a duplicate add is refused
     */
    public static void main(String[] args) {
        for (String key : animalSet.keys()) {
            System.out.println(animalSet.get(key));
        }

        // A second add of the same key is refused; stored properties stay kitten/2
        boolean added = animalSet.add("cat", Map.of("baby", "duplicate", "adult", 99));
        System.out.println("Cat is Duplicate and adding it is " + added); // false
        System.out.println("Cat dataisstill" + animalSet.get("cat"));
        System.out.println("Checking forMissing data, return values is " + animalSet.get("missing")); // null
    }
}
Catalog.main(null);


{robin, {adult=1, baby=chick}}
{cat, {adult=2, baby=kitten}}
{dog, {adult=2, baby=puppy}}
{lion, {adult=8, baby=cub}}
{pig, {adult=3, baby=piglet}}
Cat is Duplicate and adding it is false
Cat dataisstill{cat, {adult=2, baby=kitten}}
Checking forMissing data, return values is null


In [ ]:
/**
 * [TEACHING GUIDE] Concrete Implementation: Animal
 * 
 * PURPOSE:
 * This class serves as a illustrative demonstration of how to fulfill the 
 * {@link Collectable} implementation contract. It transforms abstract 
 * requirements into a working, sortable data model.
 *
 * ✅ CONTRACT FULFILLMENT CHECKLIST:
 * 
 * 1. [X] DEFINE TYPE:
 *    In the constructor, we call {@code super.setType("Animal")}. 
 *    This satisfies the requirement to categorize the data.
 *
 * 2. [X] IMPLEMENT KEY SELECTION:
 *    - We defined a concrete {@link KeyType} enum that implements {@link KeyTypes}.
 *    - We implemented {@link #getKey()} to return the current static {@code key}.
 *    This provides the "bridge" between the sorting engine and our data.
 *
 * 3. [X] OVERRIDE REPRESENTATION:
 *    We implemented {@link #toString()} with conditional logic. 
 *    It checks the active key to decide whether to show just the {@code name}, 
 *    the {@code age}, the {@code color}, or the full details.
 *
 * 💡 KEY CONCEPT: DYNAMIC IDENTITY
 * Notice that because the {@code key} is {@code static}, changing it via 
 * {@link Animal#setOrder(KeyTypes)} changes the identity and sort order 
 * of EVERY Animal instance at once. This is the power of shared state!
 */
public class Animal extends Collectable {
	// Class data
	public static KeyTypes key = KeyType.defaultKey;  // static initializer
	public static void setOrder(KeyTypes key) { Animal.key = key; }
	public enum KeyType implements KeyTypes {defaultKey, surname, name, age, color}

	// Instance data
	private final String surname;
	private final Catalog.Entry name;
	private final int age;
	private final String color;

	/* constructor
	 *
	 */
	public Animal(String surname, Catalog.Entry name, int age, String color)
	{
		super.setType("Animal");
		this.surname = surname;
		this.name = name;
		this.age = age;
		this.color = color;
	}

	/* 'Collectable' requires getKey to help enforce KeyTypes usage */
	@Override
	protected KeyTypes getKey() { return Animal.key; }

	/* Getters / Accessors
	 * 
	 */
	public String getSurname() { return this.surname; }
	public Catalog.Entry getName() { return this.name; }
	public int getAge() { return this.age; }
	public String getColor() { return this.color; }

	
	/* 'Collectable' requires toString override
	 *  toString provides data order based on Key setting
	 *  name uses Catalog.Entry.toString to include the key and properties
	 */
	@Override
	public String toString()
	{
		String output=super.getType() + ": ";
		if (KeyType.name.equals(this.getKey())) {
			output += this.name + ", " + this.surname + ", " + this.color + ", " + this.age;
		} else if (KeyType.age.equals(this.getKey())) {
			output += String.format("%02d", this.age) + ", " + this.surname + ", " + this.name + ", " + this.color;
		} else if (KeyType.color.equals(this.getKey())) {
			output += this.color + ", " + this.surname + ", " + this.name + ", " + this.age;
		} else {
			output += this.surname + ", " + this.name + ", " + this.color + ", " + this.age;
		}
		return output;
		
	}

	// Test data initializer
	public static Animal[] animals() {
		// Assign catalog to Animal class
		return new Animal[]{
				new Animal("Simba", Catalog.animalSet.get("cat"), 8, "Gold"),
				new Animal("Porky", Catalog.animalSet.get("pig"), 3, "Pink"),
				new Animal("Tweety", Catalog.animalSet.get("robin"), 7, "Red"),
				new Animal("Hobbes", Catalog.animalSet.get("cat"), 10, "Black"),
				new Animal("Mittens", Catalog.animalSet.get("cat"), 1, "Calico"),
				new Animal("Duke", Catalog.animalSet.get("dog"), 14, "Brown")
		};
	}
	
	/* main to test Animal class
	 * 
	 */
	public static void main(String[] args)
	{
		// Use class method to get test data
		Animal[] objs = Animal.animals(); 

		// print surname order, default 
		Animal.setOrder(KeyType.defaultKey);
		Animal.print(objs);

		// print in name order
		Animal.setOrder(KeyType.name);
		Animal.print(objs);

		// print in age order
		Animal.setOrder(KeyType.age);
		Animal.print(objs);
	}

}
Animal.main(null);

class [LREPL.$JShell$19CZ$Animal; 6
Collectable: Animal listed by defaultKey
Animal: Duke, {dog, {adult=2, baby=puppy}}, Brown, 14
Animal: Hobbes, {cat, {adult=2, baby=kitten}}, Black, 10
Animal: Mittens, {cat, {adult=2, baby=kitten}}, Calico, 1
Animal: Porky, {pig, {adult=3, baby=piglet}}, Pink, 3
Animal: Simba, {cat, {adult=2, baby=kitten}}, Gold, 8
Animal: Tweety, {robin, {adult=1, baby=chick}}, Red, 7

class [LREPL.$JShell$19CZ$Animal; 6
Collectable: Animal listed by name
Animal: {cat, {adult=2, baby=kitten}}, Hobbes, Black, 10
Animal: {cat, {adult=2, baby=kitten}}, Mittens, Calico, 1
Animal: {cat, {adult=2, baby=kitten}}, Simba, Gold, 8
Animal: {dog, {adult=2, baby=puppy}}, Duke, Brown, 14
Animal: {pig, {adult=3, baby=piglet}}, Porky, Pink, 3
Animal: {robin, {adult=1, baby=chick}}, Tweety, Red, 7

class [LREPL.$JShell$19CZ$Animal; 6
Collectable: Animal listed by age
Animal: 01, Mittens, {cat, {adult=2, baby=kitten}}, Calico
Animal: 03, Porky, {pig, {adult=3, baby=piglet}}, Pink
An

## Map
 Map-Based Indexing with Dynamic Views.  This pattern optimizes for high-frequency writes by separating the primary storage (Index) from the sorted representation (View).
 
```mermaid
graph TD
    subgraph index ["Source of Truth (Index)"]
        A[HashMap] -->|"O(1) Access"| B("Data Objects")
    end

    subgraph view ["Presentation Layer (View)"]
        B -->|"On-Demand Conversion"| C["Array/List"]
        C -->|"O(n log n) Sort"| D("Sorted View")
    end
```

In [133]:
import java.util.Map;
import java.util.HashMap;

/**
 * The Dual-Structure Pattern: Map-Based Indexing with Dynamic Views.
 * 
 * This implementation demonstrates a hybrid data management strategy that separates 
 * high-performance indexing from on-demand presentation.
 * 
 * 1. INDEX (HashMap): Provides $O(1)$ access, insertion, and deletion. 
 *    Acts as the primary "Source of Truth".
 * 
 * 2. VIEW (Dynamic Sorting): Generates sorted representations on-demand.
 *    Optimizes for write-heavy workloads by deferring the $O(n \log n)$ sorting 
 *    cost until a view is explicitly requested.
 *
 * @see Animal
 */
public class Registry {

    /** 
     * Primary index for $O(1)$ lookups. 
     * Uses Object to implement a heterogeneous container (Polymorphism).
     */
    private final Map<String, Object> index = new HashMap<>();

    /**
     * Adds an item to the registry.
     * 
     * @param key The unique identifier.
     * @param item The object to store.
     */
    public void add(String key, Object item) {
        index.put(key, item);
    }

    /**
     * Removes an item from the registry.
     * 
     * @param key The unique identifier.
     * @return The removed object, or null.
     */
    public Object remove(String key) {
        return index.remove(key);
    }

    /**
     * Retrieves an item from the registry.
     * 
     * @param key The unique identifier.
     * @return The object, or null.
     */
    public Object get(String key) {
        return index.get(key);
    }

    /**
     * Generates and prints a sorted view of the registry contents.
     */
    public void print() {
        // Convert Map values to an array for processing. $O(n)$ complexity.
        Collectable[] objs = index.values().toArray(new Collectable[0]);
        
        // Leverages polymorphism: Collectable.print handles the specific implementation.
        Collectable.print(objs);
    }

    /**
     * Demonstration of the Registry pattern using Animal objects.
     */
    public static void main(String[] args) {
        Registry pets = new Registry();
        Animal[] animals = Animal.animals();
        
        for (Animal animal : animals) {
            // $O(1)$ insertion via HashMap.
            pets.add(animal.getSurname(), animal);
        }
        
        // Default view (sorted by surname).
        Animal.setOrder(Animal.KeyType.defaultKey);
        pets.print();
        
        // Dynamically change sorting criteria.
        Animal.setOrder(Animal.KeyType.age);
        System.out.println("Sorted by age:");
        pets.print();
        
        Animal.setOrder(Animal.KeyType.surname);
        
        // Demonstrate removal.
        String key = "Mittens";
        Object animal = pets.remove(key);
        
        // Demonstrates inheritance: toString() is provided by java.lang.Object.
        System.out.println("Removed: " + animal);
        pets.print();
    }
}
Registry.main(null);

class [LREPL.$JShell$18G$Collectable; 6
Collectable: Animal listed by defaultKey
Animal: Duke, {dog, {adult=2, baby=puppy}}, Brown, 14
Animal: Hobbes, {cat, {adult=2, baby=kitten}}, Black, 10
Animal: Mittens, {cat, {adult=2, baby=kitten}}, Calico, 1
Animal: Porky, {pig, {adult=3, baby=piglet}}, Pink, 3
Animal: Simba, {cat, {adult=2, baby=kitten}}, Gold, 8
Animal: Tweety, {robin, {adult=1, baby=chick}}, Red, 7

Sorted by age:
class [LREPL.$JShell$18G$Collectable; 6
Collectable: Animal listed by age
Animal: 01, Mittens, {cat, {adult=2, baby=kitten}}, Calico
Animal: 03, Porky, {pig, {adult=3, baby=piglet}}, Pink
Animal: 07, Tweety, {robin, {adult=1, baby=chick}}, Red
Animal: 08, Simba, {cat, {adult=2, baby=kitten}}, Gold
Animal: 10, Hobbes, {cat, {adult=2, baby=kitten}}, Black
Animal: 14, Duke, {dog, {adult=2, baby=puppy}}, Brown

Removed: Animal: Mittens, {cat, {adult=2, baby=kitten}}, Calico, 1
class [LREPL.$JShell$18G$Collectable; 5
Collectable: Animal listed by surname
Animal: Duke, {

## Set

> Below is an example using a java.util.Set and listed are some key things to consider when using this data structure.  A Set works similarly to a key in a HashMap, a Set is just the Key.

1. No duplicates: A Set does not allow duplicate elements. If you try to add an element that already exists in the Set, the add() method will return false and the Set will not be modified.  Duplicate add is shown in example.

2. Unordered: A Set does not maintain the insertion order of elements. The order of elements in a Set may change as elements are added or removed.

3. Equality: Two Sets are considered equal if they have the same elements, regardless of their order. The equals() method is used to test for Set equality.

4. Implementation classes: Java provides several implementation classes for the Set interface, including HashSet, LinkedHashSet, and TreeSet. Each implementation has different performance characteristics and is optimized for different use cases.

5. Iterator: The iterator() method can be used to iterate over the elements in a Set. The order in which elements are returned by the iterator is not defined and may change over time as elements are added or removed from the Set. The forEach() method is another way to iterate over the elements in a Set, and it allows you to pass a lambda expression to process each element in the Set.  Lambda expression is shown in example.

## Hacks
> Analyze the Big O complexity on Sorts.
- Establish analytics including: time to sort, number of comparisons and number of swaps.
- Average the results for each each Sort, run each at least 12 times and 5000 elements.  You should throw out High and Low when doing analysis.
- Make your final/judgement on best sort: Number of Comparisons, Number of Swaps, Big O complexity, and Total Time.

> Build your own Hashmap.  Make a HashMap to correspond to a Data Structure using a Collection.
- Be sure to have 5000 records
- Perform analysis on Binary Search vs HashMap Lookup, try using random to search and find 100 keys in 5000 records.  Perform 12 times and throw out high and low.

> Extra, Practical learning
- Performing Iteration, Delete, and Add operations are another way to analyze Collection vs HashMap data structure.
- A HashMap and a Collection can be used in a Class, POJO and API.
- Make a Diagram on the Pros and Cons of Collection vs HashMap